# Fase 1 — Inventario y análisis de datasets

Este cuaderno revisa el dataset propio y MVTec D2S en una misma tabla de inventario. Las imágenes originales no se modifican.

**Estado actual:** 50 fotos de productos individuales en 11 clases provisionales y 71 fotografías de escenas de supermercado. Las escenas multiclase están en `data/propio/imagenes/` y se identifican por el prefijo `supermercado_`; D2S aún está pendiente de descarga.

## 1. Rutas y configuración
Busca la carpeta del proyecto desde el directorio actual o sus directorios padre.

In [1]:
from pathlib import Path
import pandas as pd
from PIL import Image
from IPython.display import display

candidatos = [Path.cwd(), *Path.cwd().parents]
PROYECTO = next((p for p in candidatos if (p / 'data').is_dir() and (p / 'metadata').is_dir()), None)
if PROYECTO is None:
    raise FileNotFoundError('No encuentro la raíz del proyecto. Abre el cuaderno dentro de Proyecto Integrador P1.')

RUTAS = {
    'propio': PROYECTO / 'data' / 'propio' / 'originales',
    'MVTec D2S': PROYECTO / 'data' / 'publico' / 'MVTec_D2S',
}
print('Raíz del proyecto:', PROYECTO)
print('MVTec D2S presente:', any(p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'} for p in RUTAS['MVTec D2S'].rglob('*')))

FileNotFoundError: No encuentro la raíz del proyecto. Abre el cuaderno dentro de Proyecto Integrador P1.

## 2. Inventario de imágenes
Recoge dimensiones y tamaño de archivo. Las categorías públicas y propias se mantienen separadas.

In [ ]:
EXTENSIONES = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
registros = []
for nombre_dataset, carpeta in RUTAS.items():
    archivos = sorted(p for p in carpeta.rglob('*') if p.is_file() and p.suffix.lower() in EXTENSIONES)
    for archivo in archivos:
        try:
            with Image.open(archivo) as imagen:
                ancho, alto = imagen.size
                modo = imagen.mode
            registros.append({
                'dataset': nombre_dataset,
                'archivo': archivo.name,
                'ruta_relativa': str(archivo.relative_to(PROYECTO)),
                'ancho_px': ancho,
                'alto_px': alto,
                'modo': modo,
                'tamano_kb': round(archivo.stat().st_size / 1024, 1),
            })
        except Exception as error:
            registros.append({'dataset': nombre_dataset, 'archivo': archivo.name, 'error': str(error)})
inventario = pd.DataFrame(registros)
print('Imágenes encontradas:', len(inventario))
display(inventario.head(20))

## 3. Resumen por dataset
El conteo de imágenes no equivale al conteo de clases ni al número de instancias anotadas.

In [ ]:
if inventario.empty:
    print('Aún no hay imágenes para resumir.')
else:
    resumen = (inventario.groupby('dataset')
               .agg(imagenes=('archivo', 'count'),
                    resoluciones=('ancho_px', lambda s: ', '.join(map(str, sorted(set(zip(s, inventario.loc[s.index, 'alto_px'])))))),
                    tamaño_medio_kb=('tamano_kb', 'mean'))
               .reset_index())
    resumen['tamaño_medio_kb'] = resumen['tamaño_medio_kb'].round(1)
    display(resumen)

## 4. Registro propio y tareas pendientes
La tabla captura datos conocidos. Los campos vacíos significan que el dato no está registrado; no se deben completar por suposición.

Las fotos individuales se documentan en `metadata/registro_capturas.csv`; las escenas multiclase del supermercado, en `metadata/registro_escenas_supermercado.csv`. Anotar cajas y máscaras en `data/propio/anotaciones/` y registrar rutas relativas.

## 5. Próximo paso
Solicitar MVTec D2S en la página oficial, descargar los archivos y confirmar cuáles particiones traen máscaras disponibles. Luego se añadirá visualización y auditoría de las anotaciones.